In [ ]:
!pip install logparser3

In [ ]:
import pandas as pd
from logparser.Drain import LogParser
!head -n 100000 /kaggle/input/bgl-full-log/BGL.log > /kaggle/working/BGL_100k.log
input_dir = '/kaggle/working/'  
output_dir = '/kaggle/working/dataset/'   
log_file = 'BGL_100k.log'
log_format = '<Label> <Timestamp> <Date> <Node> <Time> <NodeRepeat> <Type> <Component> <Level> <Content>'
regex = [
    r'(?<= )[0-9]+(?= )',      
    r'(?<=0x)[0-9a-fA-F]+',    
    r'([0-9]+\.){3}[0-9]+',    
    r'\b[A-Z0-9]{8}\b'         
]

st = 0.5      
depth = 4

parser = LogParser(log_format, indir=input_dir, outdir=output_dir, depth=depth, st=st, rex=regex)

parser.parse(log_file)

print("Done")

In [ ]:
df1 = pd.read_csv("/kaggle/working/dataset/BGL_100k.log_structured.csv")
df2 = pd.read_csv("/kaggle/working/dataset/BGL_100k.log_templates.csv")

df1 = df1.drop("ParameterList",axis=1)
df2 = df2.drop("Occurrences",axis=1)

unique_hashes = df1['EventId'].unique()

hash_to_e_id = {old_id: f'E{i+1}' for i, old_id in enumerate(unique_hashes)}

df1['EventId'] = df1['EventId'].map(hash_to_e_id)
df2['EventId'] = df2['EventId'].map(hash_to_e_id)

df1.to_csv('/kaggle/working/dataset/BGL_100k_log_structured_cleaned.csv', index=False)
df2.to_csv('/kaggle/working/dataset/BGL_100k_log_templates_cleaned.csv', index=False)

In [ ]:
import pandas as pd

def split(csv_path, train_ratio=0.6, val_ratio=0.2):
    df = pd.read_csv(csv_path)
    df = df.sort_values("Timestamp").reset_index(drop=True)
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    train_df = df.iloc[:train_end]
    val_df = df.iloc[train_end:val_end]
    test_df = df.iloc[val_end:]
    return df,train_df, val_df, test_df


In [ ]:
df,train_df,val_df,test_df = split("/kaggle/input/bgl-log-clean/BGL_100k_log_structured_cleaned.csv")
id_to_template = dict(zip(df["EventId"], df["EventTemplate"]))
event_ids = df["EventId"].unique().tolist()
event2id = {eid: idx for idx, eid in enumerate(event_ids)}
event2id["[MASK]"] = len(event2id)

train_df.head()

In [ ]:
print(type(id_to_template))

In [ ]:
id_to_template = dict(zip(df["EventId"], df["EventTemplate"]))

In [ ]:
id2event = {v: k for k, v in event2id.items()}

In [ ]:
import torch
import pandas as pd
from torch.utils.data import DataLoader
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WINDOW_SIZE = 10
BATCH_SIZE = 32
EPOCHS = 30

In [ ]:
import torch
from torch.utils.data import Dataset
import random

class LogDataset(Dataset):
    def __init__(self, df, window_size, event2id, mask_ratio=0.15):
        self.window_size = window_size
        self.mask_ratio = mask_ratio
        self.event2id = event2id
        self.sequences = []

        self._build_sequences(df)

    def _build_sequences(self, df):
        for node, group in df.groupby("Node"):
            events = group.sort_values("Timestamp")["EventId"].tolist()

            if len(events) <= self.window_size:
                continue

            for i in range(len(events) - self.window_size):
                self.sequences.append(events[i:i+self.window_size])


In [ ]:
import random

all_dataset = LogDataset(df, WINDOW_SIZE, event2id, mask_ratio=0.15)
all_sequences = all_dataset.sequences

random.shuffle(all_sequences)

n = len(all_sequences)
train_seq = all_sequences[:int(0.6*n)]
val_seq   = all_sequences[int(0.6*n):int(0.8*n)]
test_seq  = all_sequences[int(0.8*n):]


In [ ]:
class SeqDataset(Dataset):
    def __init__(self, sequences, event2id, mask_ratio):
        self.sequences = sequences
        self.event2id = event2id
        self.mask_ratio = mask_ratio

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        input_ids = [self.event2id[e] for e in seq]
        original_ids = [self.event2id[e] for e in seq]
        labels = [-100] * len(input_ids)

        for i in range(len(input_ids)):
            if random.random() < self.mask_ratio:
                labels[i] = input_ids[i]
                input_ids[i] = self.event2id["[MASK]"]

        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(labels, dtype=torch.long),
            torch.tensor(original_ids, dtype=torch.long)  
            )


In [ ]:
train_dataset = SeqDataset(train_seq, event2id, mask_ratio=0.15)
val_dataset   = SeqDataset(val_seq,   event2id, mask_ratio=0.15)

print(len(train_dataset), len(val_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
import torch
import torch.nn as nn

class LogBERT(nn.Module):
    def __init__(self, vocab_size, hidden_size=64, num_layers=1, num_heads=2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, hidden_size)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=num_heads,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        out = self.encoder(emb)
        return self.fc(out)


In [ ]:
model = LogBERT(len(event2id)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)

In [ ]:
val_losses = []
val_scores = []
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for x, y,_ in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(
            logits.view(-1, logits.size(-1)),
            y.view(-1)
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y,_ in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = criterion(
                logits.view(-1, logits.size(-1)),
                y.view(-1)
            )
            val_loss += loss.item()
            sep_loss = torch.nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)), 
                y.view(-1), 
                reduction='none', 
                ignore_index=-100
            )
            sep_loss = sep_loss.view(x.size(0), -1).mean(dim=1)
            val_scores.extend(sep_loss.cpu().tolist())
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} | "f"Train Loss: {train_loss:.4f} | "f"Val Loss: {val_loss:.4f}")
    

In [ ]:
import statistics
import numpy as np
thresh = np.percentile(val_scores, 95)

print(thresh)

In [ ]:
torch.save(model.state_dict(), "logbert.pt")

In [ ]:
test_event_ids = test_df["EventId"].unique().tolist()
test_event2id = {e: i for i, e in enumerate(event_ids)}
test_event2id["[MASK]"] = len(event2id)

In [ ]:
test_dataset = SeqDataset(test_seq, event2id, mask_ratio=0.15)
test_loader = DataLoader(test_dataset,   batch_size=1, shuffle=False)
model_test = LogBERT(len(event2id)).to(DEVICE)
model_test.load_state_dict(torch.load("logbert.pt"))
model_test.eval()

In [ ]:
print("Test sequences:", len(test_dataset))
print("Test batches:", len(test_loader))

In [ ]:
import json
anomalies = []

criterion_token = torch.nn.CrossEntropyLoss(reduction="none")
print(f"Length of test_loader:{len(test_loader)}")
with torch.no_grad():
    for idx, (input_ids, labels,original_ids) in enumerate(test_loader):
        input_ids = input_ids.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model_test(input_ids)  
        

        loss_per_token = criterion_token(
            logits.view(-1, logits.size(-1)),  
            labels.view(-1)                    
        )  

        mask = labels.view(-1) != -100
        if mask.sum() == 0:
            continue

        masked_loss = loss_per_token[mask]
        anomaly_score = masked_loss.mean().item()
        
        #print(anomaly_score)
        if anomaly_score > thresh: 
            orig_tokens = original_ids.cpu().tolist()[0]
            event_ids = [id2event[t] for t in orig_tokens]
            anomalies.append({
                "sequence_id": idx,
                "anomaly_score": anomaly_score,
                "events": event_ids
            })


In [ ]:
print(len(anomalies))

In [ ]:
for anomaly in anomalies:
    anomaly["events"] = [
        id_to_template[eid] for eid in anomaly["events"]
    ]

In [ ]:
with open("/kaggle/working/anomalies5.json", "w") as f:
    json.dump(anomalies, f, indent=2)

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/input/bgl-log-clean/BGL_100k_log_structured_cleaned.csv")
print(len(df))

#### Fine Tuning LLMs


In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -U transformers peft accelerate bitsandbytes accelerate

In [2]:
def format_instruction(sample):
    instruction = "Analyze the following log sequence and determine if it is normal or abnormal. Provide a concise explanation for your judgment."
    input_text = f"Input: {sample['log_seq']}"
    output_text = f"Output: {sample['label-cause pair']}"
    return f"### Instruction:\n{instruction}\n\n### {input_text}\n\n### {output_text}"

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

model_id = "mistralai/Mistral-7B-Instruct-v0.2"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
peft_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], 
    lora_dropout=0.05, 
    bias="none", 
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [4]:
from transformers import Trainer, TrainingArguments
import torch

In [5]:
import ast
with open('/kaggle/input/datasets/djshadow/ft-dataset/seed_bgl.json', 'r') as f:
    raw_data = ast.literal_eval(f.read())
formatted_data = [format_instruction(item) for item in raw_data]
from datasets import Dataset
dataset = Dataset.from_dict({"text": formatted_data})
print(dataset[0]['text'])

### Instruction:
Analyze the following log sequence and determine if it is normal or abnormal. Provide a concise explanation for your judgment.

### Input: quiet <*>................................<*>,minus inf................................<*>,minus normalized number..................<*>,minus denormalized number................<*>,plus zero................................<*>,plus denormalized number.................<*>,plus normalized number...................<*>,plus infinity............................<*>,reserved.................................<*>,invalid operation exception (software)...<*>,invalid operation exception (sqrt).......<*>,invalid operation exception (int cnvt)...<*>,enable invalid operation exceptions......<*>,enable overflow exceptions...............<*>,enable underflow exceptions..............<*>,enable divide-by-zero exceptions.........<*>,enable inexact exceptions................<*>,enable non-IEEE mode.....................<*>,round nearest.....................

In [6]:
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"], 
        truncation=True, 
        max_length=512, 
        padding="max_length"
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized
tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

In [7]:
import torch.nn as nn
class LLMLADELoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=2.0):
        super().__init__()
        self.alpha = alpha  
        self.beta = beta    
        self.gamma = gamma  
        self.ce_loss = nn.CrossEntropyLoss(reduction='none')

    def forward(self, logits, labels):
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        logits_flat = shift_logits.view(-1, shift_logits.size(-1))
        labels_flat = shift_labels.view(-1)
        raw_loss = self.ce_loss(logits_flat, labels_flat)
        probs = torch.softmax(logits_flat, dim=-1)
        pt = probs.gather(1, labels_flat.unsqueeze(1)).squeeze(1)
        focal_loss = (self.alpha * (1 - pt)**self.gamma * raw_loss)
        cause_loss = (self.beta * raw_loss)
        return focal_loss.mean() + cause_loss.mean()

In [8]:
class LLMLADETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False,**kwargs):
        outputs = model(**inputs)
        logits = outputs.get("logits")
        labels = inputs.get("labels")
        
        loss_fct = LLMLADELoss(alpha=0.7, beta=0.3, gamma=2.0)
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

In [9]:
training_args = TrainingArguments(
    output_dir="./llm_lade_results",
    num_train_epochs=1,               
    per_device_train_batch_size=1,    
    gradient_accumulation_steps=8,    
    learning_rate=5e-5,              
    lr_scheduler_type="cosine",       
    optim="adamw_torch",             
    fp16=True,
    logging_steps=10,
)
trainer = LLMLADETrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)
trainer.train()

Step,Training Loss


TrainOutput(global_step=4, training_loss=23.311342239379883, metrics={'train_runtime': 49.7089, 'train_samples_per_second': 0.604, 'train_steps_per_second': 0.08, 'total_flos': 656574712381440.0, 'train_loss': 23.311342239379883, 'epoch': 1.0})

In [10]:
trainer.save_model("./llm_lade_results")

In [11]:
input_file = "/kaggle/working/anomalies5.json"
output_file = "/kaggle/working/llm_results.json"

In [13]:
import torch
import gc

# Delete any existing model variables if they exist
if 'model' in locals(): del model
if 'base_model' in locals(): del base_model
if 'trainer' in locals(): del trainer

# Force garbage collection
gc.collect()

# Clear the PyTorch cache
torch.cuda.empty_cache()

# Optional: verify how much memory is free
free_memory = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
print(f"Free GPU memory: {free_memory / 1024**3:.2f} GB")

Free GPU memory: 10.70 GB


In [14]:
from peft import PeftModel
import os
base_model_path = "mistralai/Mistral-7B-Instruct-v0.2" 
adapter_path = os.path.abspath("./llm_lade_results")     
tokenizer = AutoTokenizer.from_pretrained(base_model_path)
tokenizer.pad_token = tokenizer.eos_token
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" 
)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_pro

In [15]:
import json
with open(input_file, 'r') as f:
    anomalies = json.load(f)

In [17]:
from tqdm import tqdm
finals = []
with torch.inference_mode():
    for entry in tqdm(anomalies):
        # Format the log events
        log_seq_str = "\n".join(entry['events'])
        
        # LLM-LADE Prompt Template
        instruction = "Analyze the following log sequence and determine if it is normal or abnormal. Provide a concise explanation for your judgment."
        prompt = f"### Instruction:\n{instruction}\n\n### Input: {log_seq_str}\n\n### Output:"

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        # Generate Verdict
        outputs = model.generate(
            **inputs, 
            max_new_tokens=128, 
            do_sample=False,
            temperature=0.0
        )
        
        full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        verdict = full_output.split("### Output:")[-1].strip()
        finals.append({
            "sequence_id": entry["sequence_id"],
            "anomaly_score": entry["anomaly_score"],
            "llm_verdict": verdict
        })   

100%|██████████| 437/437 [1:12:00<00:00,  9.89s/it]


NameError: name 'final_results' is not defined

In [19]:
with open(output_file, 'w') as f:
    json.dump(finals, f, indent=4)     